In [32]:
import warnings
from neurokit2 import NeuroKitWarning
warnings.simplefilter("ignore", NeuroKitWarning)

import neurokit2 as nk
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import pywt

## Read data

In [3]:
original_df = pd.read_csv('./merged_data.csv', low_memory=False)
original_df

,X,Y,Z,EDA,HR,TEMP,id,datetime,label
0,-13.0,-61.0,5.0,6.769995,99.43,31.17,15,2020-07-08 14:03:00.000000000,2.0
1,-20.0,-69.0,-3.0,6.769995,99.43,31.17,15,2020-07-08 14:03:00.031249920,2.0
2,-31.0,-78.0,-15.0,6.769995,99.43,31.17,15,2020-07-08 14:03:00.062500096,2.0
3,-47.0,-65.0,-38.0,6.769995,99.43,31.17,15,2020-07-08 14:03:00.093750016,2.0
4,-67.0,-57.0,-53.0,6.769995,99.43,31.17,15,2020-07-08 14:03:00.124999936,2.0
...,...,...,...,...,...,...,...,...,...
11509046,-16.0,-56.0,24.0,3.386070,88.37,33.77,F5,2020-07-23 17:28:59.875000064,2.0
11509047,-8.0,-50.0,27.0,3.386070,88.37,33.77,F5,2020-07-23 17:28:59.906249984,2.0
11509048,-28.0,-36.0,28.0,3.386070,88.37,33.77,F5,2020-07-23 17:28:59.937499904,2.0
11509049,-29.0,-29.0,30.0,3.386070,88.37,33.77,F5,2020-07-23 17:28:59.968750080,2.0


In [4]:
original_df = original_df[original_df["EDA"] != 0]

In [5]:
participants = original_df["id"].unique()

## Split data into time-series segments

In [6]:
new_df = original_df
# Ensure the datetime column is in datetime format
new_df['datetime'] = pd.to_datetime(new_df['datetime'])

# Initialize an empty list to store the results
all_segments = []

# Group the data by "id"
grouped = new_df.groupby('id')

# Define the threshold for identifying gaps (e.g., 0.1 seconds)
threshold = 1.0

# Process each group separately
for _, group in grouped:
    group = group.sort_values(by='datetime').copy()

    # Calculate the time difference between consecutive rows
    group['time_diff'] = group['datetime'].diff().dt.total_seconds()

    # Handle first row which will have NaN time_diff
    group['time_diff'] = group['time_diff'].fillna(0)

    # Identify gaps
    group['gap'] = (group['time_diff'] > threshold)

    # Create a segment ID based on gaps
    group['segment_id'] = group['gap'].cumsum()

    # Drop unnecessary columns
    group = group.drop(columns=['time_diff', 'gap'])

    # Group by segment_id and assign labels
    segments = []
    for seg_id, segment in group.groupby('segment_id'):
        segment_copy = segment.copy()
        segment_copy['label'] = segment_copy['label'].iloc[0]
        segments.append(segment_copy)

    # Append the processed segments to the list
    all_segments.append(pd.concat(segments))

# Combine all segments back into a single dataframe
df_with_time_series_index = pd.concat(all_segments)
df_with_time_series_index.reset_index(drop=True, inplace=True)
df_with_time_series_index

/var/folders/ng/n1v_1pr17m73rptx7l3q3hz40000gr/T/ipykernel_4670/628560351.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['datetime'] = pd.to_datetime(new_df['datetime'])


,X,Y,Z,EDA,HR,TEMP,id,datetime,label,segment_id
0,-31.0,-46.0,8.0,2.567307,77.27,32.65,15,2020-07-08 13:09:00.000000000,2.0,0
1,-34.0,-48.0,5.0,2.567307,77.27,32.65,15,2020-07-08 13:09:00.031249920,2.0,0
2,-39.0,-51.0,0.0,2.567307,77.27,32.65,15,2020-07-08 13:09:00.062500096,2.0,0
3,-44.0,-54.0,-2.0,2.567307,77.27,32.65,15,2020-07-08 13:09:00.093750016,2.0,0
4,-47.0,-56.0,-8.0,2.567307,77.27,32.65,15,2020-07-08 13:09:00.124999936,2.0,0
...,...,...,...,...,...,...,...,...,...,...
11477942,-16.0,-56.0,24.0,3.386070,88.37,33.77,F5,2020-07-23 17:28:59.875000064,2.0,39
11477943,-8.0,-50.0,27.0,3.386070,88.37,33.77,F5,2020-07-23 17:28:59.906249984,2.0,39
11477944,-28.0,-36.0,28.0,3.386070,88.37,33.77,F5,2020-07-23 17:28:59.937499904,2.0,39
11477945,-29.0,-29.0,30.0,3.386070,88.37,33.77,F5,2020-07-23 17:28:59.968750080,2.0,39


## Feature Engineering

In [7]:
def strided_app(a, L, S ):  # Window len = L, Stride len/stepsize = S
    nrows = ((a.size-L)//S)+1
    n = a.strides[0]
    return np.lib.stride_tricks.as_strided(a, shape=(nrows,L), strides=(S*n,n))

In [30]:
from biosppy.signals import tools
from typing import Any


def get_statistics(
    data: np.ndarray,
) -> tuple[Any, Any, Any, Any, Any, Any, Any, Any, Any, Any, Any, Any]:
    s_mean, s_median, s_min, s_max, s_max_amp, s_range, s_var, s_std_dev, s_abs_dev, s_rms, s_kurtosis, s_skew = (
        tools.signal_stats(data)
    )
    return (
        s_mean,
        s_median,
        s_min,
        s_max,
        s_max_amp,
        s_range,
        s_var,
        s_std_dev,
        s_abs_dev,
        s_rms,
        s_kurtosis,
        s_skew,
    )

In [31]:
def get_derivatives(data: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    # Get the first and second derivatives of the data
    deriv = (data[1:-1] + data[2:]) / 2.0 - (data[1:-1] + data[:-2]) / 2.0
    second_deriv = data[2:] - 2 * data[1:-1] + data[:-2]
    return deriv, second_deriv

In [ ]:
final_data = []  # Store rows in a list instead of repeatedly concatenating DataFrames

for participant in tqdm(participants, desc=" Participants", position=0):
    participant_df = df_with_time_series_index[df_with_time_series_index['id'] == participant]

    for segment_id in tqdm(participant_df['segment_id'].unique(), desc=" Segments", position=1, leave=False):
        segment_df = participant_df[participant_df['segment_id'] == segment_id]
        label = segment_df["label"].iloc[0]

        if len(segment_df) < 1920:
            continue

        window_length = 1920 # 60s at 32Hz
        step_size = 8

        eda_windows = strided_app(segment_df["EDA"].values, window_length, 8)
        temp_windows = strided_app(segment_df["TEMP"].values, window_length, 8)
        hr_windows = strided_app(segment_df["HR"].values, window_length, 8)
        acc_x_windows = strided_app(segment_df["X"].values, window_length, 8)
        acc_y_windows = strided_app(segment_df["Y"].values, window_length, 8)
        acc_z_windows = strided_app(segment_df["Z"].values, window_length, 8)

        # Iterate over all the windows
        for i in tqdm(range(len(eda_windows)), desc=" Windows", position=2, leave=False):
            deriv_EDA, second_deriv_EDA = get_derivatives(data=eda_windows[i])
            deriv_HR, second_deriv_HR = get_derivatives(data=hr_windows[i])
            deriv_TEMP, second_deriv_TEMP = get_derivatives(data=temp_windows[i])
            deriv_ACC_X, second_deriv_ACC_X = get_derivatives(data=acc_x_windows[i])
            deriv_ACC_Y, second_deriv_ACC_Y = get_derivatives(data=acc_y_windows[i])
            deriv_ACC_Z, second_deriv_ACC_Z = get_derivatives(data=acc_z_windows[i])

            _, EDA_cD_3, EDA_cD_2, EDA_cD_1 = pywt.wavedec(eda_windows[i], "Haar", level=3)  # 3 = 1Hz, 2 = 2Hz, 1=4Hz
            _, HR_cD_3, HR_cD_2, HR_cD_1 = pywt.wavedec(hr_windows[i], "Haar", level=3)  # 3 = 1Hz, 2 = 2Hz, 1=4Hz
            _, TEMP_cD_3, TEMP_cD_2, TEMP_cD_1 = pywt.wavedec(temp_windows[i], "Haar", level=3)  # 3 = 1Hz, 2 = 2Hz, 1=4Hz
            _, ACC_X_cD_3, ACC_X_cD_2, ACC_X_cD_1 = pywt.wavedec(acc_x_windows[i], "Haar", level=3)  # 3 = 1Hz, 2 = 2Hz, 1=4Hz
            _, ACC_Y_cD_3, ACC_Y_cD_2, ACC_Y_cD_1 = pywt.wavedec(acc_y_windows[i], "Haar", level=3)  # 3 = 1Hz, 2 = 2Hz, 1=4Hz
            _, ACC_Z_cD_3, ACC_Z_cD_2, ACC_Z_cD_1 = pywt.wavedec(acc_z_windows[i], "Haar", level=3)  # 3 = 1Hz, 2 = 2Hz, 1=4Hz

            # Analyze EDA with NeuroKit2
            processed_data, info = nk.eda_process(eda_windows[i], sampling_rate=4)
            results = nk.eda_analyze(processed_data, sampling_rate=4)

            # Store data as dictionary (faster than creating DataFrame inside loop)
            row = {"Participant": participant, "Label": label, **results.to_dict(orient="records")[0]}

            # ----- EDA features -----
            # EDA statistical features:
            (
                row["EDA_mean"],
                row["EDA_median"],
                row["EDA_min"],
                row["EDA_max"],
                row["EDA_max_amp"],
                row["EDA_range"],
                row["EDA_var"],
                row["EDA_std_dev"],
                row["EDA_abs_dev"],
                row["EDA_rms"],
                row["EDA_kurtosis"],
                row["EDA_skew"],
            ) = get_statistics(data=eda_windows[i])
            (
                row["Deriv_EDA_mean"],
                row["Deriv_EDA_median"],
                row["Deriv_EDA_min"],
                row["Deriv_EDA_max"],
                row["Deriv_EDA_max_amp"],
                row["Deriv_EDA_range"],
                row["Deriv_EDA_var"],
                row["Deriv_EDA_std_dev"],
                row["Deriv_EDA_abs_dev"],
                row["Deriv_EDA_rms"],
                row["Deriv_EDA_kurtosis"],
                row["Deriv_EDA_skew"],
            ) = get_statistics(data=deriv_EDA)
            (
                row["Deriv_2_EDA_mean"],
                row["Deriv_2_EDA_median"],
                row["Deriv_2_EDA_min"],
                row["Deriv_2_EDA_max"],
                row["Deriv_2_EDA_max_amp"],
                row["Deriv_2_EDA_range"],
                row["Deriv_2_EDA_var"],
                row["Deriv_2_EDA_std_dev"],
                row["Deriv_2_EDA_abs_dev"],
                row["Deriv_2_EDA_rms"],
                row["Deriv_2_EDA_kurtosis"],
                row["Deriv_2_EDA_skew"],
            ) = get_statistics(data=second_deriv_EDA)
            # EDA wavelet features:
            (
                row["Wavelet_1Hz_EDA_mean"],
                row["Wavelet_1Hz_EDA_median"],
                row["Wavelet_1Hz_EDA_min"],
                row["Wavelet_1Hz_EDA_max"],
                row["Wavelet_1Hz_EDA_max_amp"],
                row["Wavelet_1Hz_EDA_range"],
                row["Wavelet_1Hz_EDA_var"],
                row["Wavelet_1Hz_EDA_std_dev"],
                row["Wavelet_1Hz_EDA_abs_dev"],
                row["Wavelet_1Hz_EDA_rms"],
                row["Wavelet_1Hz_EDA_kurtosis"],
                row["Wavelet_1Hz_EDA_skew"],
            ) = get_statistics(data=EDA_cD_3)
            (
                row["Wavelet_2Hz_EDA_mean"],
                row["Wavelet_2Hz_EDA_median"],
                row["Wavelet_2Hz_EDA_min"],
                row["Wavelet_2Hz_EDA_max"],
                row["Wavelet_2Hz_EDA_max_amp"],
                row["Wavelet_2Hz_EDA_range"],
                row["Wavelet_2Hz_EDA_var"],
                row["Wavelet_2Hz_EDA_std_dev"],
                row["Wavelet_2Hz_EDA_abs_dev"],
                row["Wavelet_2Hz_EDA_rms"],
                row["Wavelet_2Hz_EDA_kurtosis"],
                row["Wavelet_2Hz_EDA_skew"],
            ) = get_statistics(data=EDA_cD_2)
            (
                row["Wavelet_4Hz_EDA_mean"],
                row["Wavelet_4Hz_EDA_median"],
                row["Wavelet_4Hz_EDA_min"],
                row["Wavelet_4Hz_EDA_max"],
                row["Wavelet_4Hz_EDA_max_amp"],
                row["Wavelet_4Hz_EDA_range"],
                row["Wavelet_4Hz_EDA_var"],
                row["Wavelet_4Hz_EDA_std_dev"],
                row["Wavelet_4Hz_EDA_abs_dev"],
                row["Wavelet_4Hz_EDA_rms"],
                row["Wavelet_4Hz_EDA_kurtosis"],
                row["Wavelet_4Hz_EDA_skew"],
            ) = get_statistics(data=EDA_cD_1)

            # ----- HR features -----
            # HR statistical features:
            (
                row["HR_mean"],
                row["HR_median"],
                row["HR_min"],
                row["HR_max"],
                row["HR_max_amp"],
                row["HR_range"],
                row["HR_var"],
                row["HR_std_dev"],
                row["HR_abs_dev"],
                row["HR_rms"],
                row["HR_kurtosis"],
                row["HR_skew"],
            ) = get_statistics(data=hr_windows[i])
            (
                row["Deriv_HR_mean"],
                row["Deriv_HR_median"],
                row["Deriv_HR_min"],
                row["Deriv_HR_max"],
                row["Deriv_HR_max_amp"],
                row["Deriv_HR_range"],
                row["Deriv_HR_var"],
                row["Deriv_HR_std_dev"],
                row["Deriv_HR_abs_dev"],
                row["Deriv_HR_rms"],
                row["Deriv_HR_kurtosis"],
                row["Deriv_HR_skew"],
            ) = get_statistics(data=deriv_HR)
            (
                row["Deriv_2_HR_mean"],
                row["Deriv_2_HR_median"],
                row["Deriv_2_HR_min"],
                row["Deriv_2_HR_max"],
                row["Deriv_2_HR_max_amp"],
                row["Deriv_2_HR_range"],
                row["Deriv_2_HR_var"],
                row["Deriv_2_HR_std_dev"],
                row["Deriv_2_HR_abs_dev"],
                row["Deriv_2_HR_rms"],
                row["Deriv_2_HR_kurtosis"],
                row["Deriv_2_HR_skew"],
            ) = get_statistics(data=second_deriv_HR)
            # HR wavelet features:
            (
                row["Wavelet_1Hz_HR_mean"],
                row["Wavelet_1Hz_HR_median"],
                row["Wavelet_1Hz_HR_min"],
                row["Wavelet_1Hz_HR_max"],
                row["Wavelet_1Hz_HR_max_amp"],
                row["Wavelet_1Hz_HR_range"],
                row["Wavelet_1Hz_HR_var"],
                row["Wavelet_1Hz_HR_std_dev"],
                row["Wavelet_1Hz_HR_abs_dev"],
                row["Wavelet_1Hz_HR_rms"],
                row["Wavelet_1Hz_HR_kurtosis"],
                row["Wavelet_1Hz_HR_skew"],
            ) = get_statistics(data=HR_cD_3)
            (
                row["Wavelet_2Hz_HR_mean"],
                row["Wavelet_2Hz_HR_median"],
                row["Wavelet_2Hz_HR_min"],
                row["Wavelet_2Hz_HR_max"],
                row["Wavelet_2Hz_HR_max_amp"],
                row["Wavelet_2Hz_HR_range"],
                row["Wavelet_2Hz_HR_var"],
                row["Wavelet_2Hz_HR_std_dev"],
                row["Wavelet_2Hz_HR_abs_dev"],
                row["Wavelet_2Hz_HR_rms"],
                row["Wavelet_2Hz_HR_kurtosis"],
                row["Wavelet_2Hz_HR_skew"],
            ) = get_statistics(data=HR_cD_2)
            (
                row["Wavelet_4Hz_HR_mean"],
                row["Wavelet_4Hz_HR_median"],
                row["Wavelet_4Hz_HR_min"],
                row["Wavelet_4Hz_HR_max"],
                row["Wavelet_4Hz_HR_max_amp"],
                row["Wavelet_4Hz_HR_range"],
                row["Wavelet_4Hz_HR_var"],
                row["Wavelet_4Hz_HR_std_dev"],
                row["Wavelet_4Hz_HR_abs_dev"],
                row["Wavelet_4Hz_HR_rms"],
                row["Wavelet_4Hz_HR_kurtosis"],
                row["Wavelet_4Hz_HR_skew"],
            ) = get_statistics(data=HR_cD_1)

            # ----- TEMP features -----
            # TEMP statistical features:
            (
                row["TEMP_mean"],
                row["TEMP_median"],
                row["TEMP_min"],
                row["TEMP_max"],
                row["TEMP_max_amp"],
                row["TEMP_range"],
                row["TEMP_var"],
                row["TEMP_std_dev"],
                row["TEMP_abs_dev"],
                row["TEMP_rms"],
                row["TEMP_kurtosis"],
                row["TEMP_skew"],
            ) = get_statistics(data=temp_windows[i])
            (
                row["Deriv_TEMP_mean"],
                row["Deriv_TEMP_median"],
                row["Deriv_TEMP_min"],
                row["Deriv_TEMP_max"],
                row["Deriv_TEMP_max_amp"],
                row["Deriv_TEMP_range"],
                row["Deriv_TEMP_var"],
                row["Deriv_TEMP_std_dev"],
                row["Deriv_TEMP_abs_dev"],
                row["Deriv_TEMP_rms"],
                row["Deriv_TEMP_kurtosis"],
                row["Deriv_TEMP_skew"],
            ) = get_statistics(data=deriv_TEMP)
            (
                row["Deriv_2_TEMP_mean"],
                row["Deriv_2_TEMP_median"],
                row["Deriv_2_TEMP_min"],
                row["Deriv_2_TEMP_max"],
                row["Deriv_2_TEMP_max_amp"],
                row["Deriv_2_TEMP_range"],
                row["Deriv_2_TEMP_var"],
                row["Deriv_2_TEMP_std_dev"],
                row["Deriv_2_TEMP_abs_dev"],
                row["Deriv_2_TEMP_rms"],
                row["Deriv_2_TEMP_kurtosis"],
                row["Deriv_2_TEMP_skew"],
            ) = get_statistics(data=second_deriv_TEMP)

            # TEMP wavelet features:
            (
                row["Wavelet_1Hz_TEMP_mean"],
                row["Wavelet_1Hz_TEMP_median"],
                row["Wavelet_1Hz_TEMP_min"],
                row["Wavelet_1Hz_TEMP_max"],
                row["Wavelet_1Hz_TEMP_max_amp"],
                row["Wavelet_1Hz_TEMP_range"],
                row["Wavelet_1Hz_TEMP_var"],
                row["Wavelet_1Hz_TEMP_std_dev"],
                row["Wavelet_1Hz_TEMP_abs_dev"],
                row["Wavelet_1Hz_TEMP_rms"],
                row["Wavelet_1Hz_TEMP_kurtosis"],
                row["Wavelet_1Hz_TEMP_skew"],
            ) = get_statistics(data=TEMP_cD_3)
            (
                row["Wavelet_2Hz_TEMP_mean"],
                row["Wavelet_2Hz_TEMP_median"],
                row["Wavelet_2Hz_TEMP_min"],
                row["Wavelet_2Hz_TEMP_max"],
                row["Wavelet_2Hz_TEMP_max_amp"],
                row["Wavelet_2Hz_TEMP_range"],
                row["Wavelet_2Hz_TEMP_var"],
                row["Wavelet_2Hz_TEMP_std_dev"],
                row["Wavelet_2Hz_TEMP_abs_dev"],
                row["Wavelet_2Hz_TEMP_rms"],
                row["Wavelet_2Hz_TEMP_kurtosis"],
                row["Wavelet_2Hz_TEMP_skew"],
            ) = get_statistics(data=TEMP_cD_2)
            (
                row["Wavelet_4Hz_TEMP_mean"],
                row["Wavelet_4Hz_TEMP_median"],
                row["Wavelet_4Hz_TEMP_min"],
                row["Wavelet_4Hz_TEMP_max"],
                row["Wavelet_4Hz_TEMP_max_amp"],
                row["Wavelet_4Hz_TEMP_range"],
                row["Wavelet_4Hz_TEMP_var"],
                row["Wavelet_4Hz_TEMP_std_dev"],
                row["Wavelet_4Hz_TEMP_abs_dev"],
                row["Wavelet_4Hz_TEMP_rms"],
                row["Wavelet_4Hz_TEMP_kurtosis"],
                row["Wavelet_4Hz_TEMP_skew"],
            ) = get_statistics(data=TEMP_cD_1)

            # ----- ACC features -----
            # ACC statistical features:
            (
                row["ACC_X_mean"],
                row["ACC_X_median"],
                row["ACC_X_min"],
                row["ACC_X_max"],
                row["ACC_X_max_amp"],
                row["ACC_X_range"],
                row["ACC_X_var"],
                row["ACC_X_std_dev"],
                row["ACC_X_abs_dev"],
                row["ACC_X_rms"],
                row["ACC_X_kurtosis"],
                row["ACC_X_skew"],
            ) = get_statistics(data=acc_x_windows[i])
            (
                row["Deriv_ACC_X_mean"],
                row["Deriv_ACC_X_median"],
                row["Deriv_ACC_X_min"],
                row["Deriv_ACC_X_max"],
                row["Deriv_ACC_X_max_amp"],
                row["Deriv_ACC_X_range"],
                row["Deriv_ACC_X_var"],
                row["Deriv_ACC_X_std_dev"],
                row["Deriv_ACC_X_abs_dev"],
                row["Deriv_ACC_X_rms"],
                row["Deriv_ACC_X_kurtosis"],
                row["Deriv_ACC_X_skew"],
            ) = get_statistics(data=deriv_ACC_X)
            (
                row["Deriv_2_ACC_X_mean"],
                row["Deriv_2_ACC_X_median"],
                row["Deriv_2_ACC_X_min"],
                row["Deriv_2_ACC_X_max"],
                row["Deriv_2_ACC_X_max_amp"],
                row["Deriv_2_ACC_X_range"],
                row["Deriv_2_ACC_X_var"],
                row["Deriv_2_ACC_X_std_dev"],
                row["Deriv_2_ACC_X_abs_dev"],
                row["Deriv_2_ACC_X_rms"],
                row["Deriv_2_ACC_X_kurtosis"],
                row["Deriv_2_ACC_X_skew"],
            ) = get_statistics(data=second_deriv_ACC_X)
            # ACC wavelet features:
            (
                row["Wavelet_1Hz_ACC_X_mean"],
                row["Wavelet_1Hz_ACC_X_median"],
                row["Wavelet_1Hz_ACC_X_min"],
                row["Wavelet_1Hz_ACC_X_max"],
                row["Wavelet_1Hz_ACC_X_max_amp"],
                row["Wavelet_1Hz_ACC_X_range"],
                row["Wavelet_1Hz_ACC_X_var"],
                row["Wavelet_1Hz_ACC_X_std_dev"],
                row["Wavelet_1Hz_ACC_X_abs_dev"],
                row["Wavelet_1Hz_ACC_X_rms"],
                row["Wavelet_1Hz_ACC_X_kurtosis"],
                row["Wavelet_1Hz_ACC_X_skew"],
            ) = get_statistics(data=ACC_X_cD_3)
            (
                row["Wavelet_2Hz_ACC_X_mean"],
                row["Wavelet_2Hz_ACC_X_median"],
                row["Wavelet_2Hz_ACC_X_min"],
                row["Wavelet_2Hz_ACC_X_max"],
                row["Wavelet_2Hz_ACC_X_max_amp"],
                row["Wavelet_2Hz_ACC_X_range"],
                row["Wavelet_2Hz_ACC_X_var"],
                row["Wavelet_2Hz_ACC_X_std_dev"],
                row["Wavelet_2Hz_ACC_X_abs_dev"],
                row["Wavelet_2Hz_ACC_X_rms"],
                row["Wavelet_2Hz_ACC_X_kurtosis"],
                row["Wavelet_2Hz_ACC_X_skew"],
            ) = get_statistics(data=ACC_X_cD_2)
            (
                row["Wavelet_4Hz_ACC_X_mean"],
                row["Wavelet_4Hz_ACC_X_median"],
                row["Wavelet_4Hz_ACC_X_min"],
                row["Wavelet_4Hz_ACC_X_max"],
                row["Wavelet_4Hz_ACC_X_max_amp"],
                row["Wavelet_4Hz_ACC_X_range"],
                row["Wavelet_4Hz_ACC_X_var"],
                row["Wavelet_4Hz_ACC_X_std_dev"],
                row["Wavelet_4Hz_ACC_X_abs_dev"],
                row["Wavelet_4Hz_ACC_X_rms"],
                row["Wavelet_4Hz_ACC_X_kurtosis"],
                row["Wavelet_4Hz_ACC_X_skew"],
            ) = get_statistics(data=ACC_X_cD_1)

            (
                row["ACC_Y_mean"],
                row["ACC_Y_median"],
                row["ACC_Y_min"],
                row["ACC_Y_max"],
                row["ACC_Y_max_amp"],
                row["ACC_Y_range"],
                row["ACC_Y_var"],
                row["ACC_Y_std_dev"],
                row["ACC_Y_abs_dev"],
                row["ACC_Y_rms"],
                row["ACC_Y_kurtosis"],
                row["ACC_Y_skew"],
            ) = get_statistics(data=acc_y_windows[i])
            (
                row["Deriv_ACC_Y_mean"],
                row["Deriv_ACC_Y_median"],
                row["Deriv_ACC_Y_min"],
                row["Deriv_ACC_Y_max"],
                row["Deriv_ACC_Y_max_amp"],
                row["Deriv_ACC_Y_range"],
                row["Deriv_ACC_Y_var"],
                row["Deriv_ACC_Y_std_dev"],
                row["Deriv_ACC_Y_abs_dev"],
                row["Deriv_ACC_Y_rms"],
                row["Deriv_ACC_Y_kurtosis"],
                row["Deriv_ACC_Y_skew"],
            ) = get_statistics(data=deriv_ACC_Y)
            (
                row["Deriv_2_ACC_Y_mean"],
                row["Deriv_2_ACC_Y_median"],
                row["Deriv_2_ACC_Y_min"],
                row["Deriv_2_ACC_Y_max"],
                row["Deriv_2_ACC_Y_max_amp"],
                row["Deriv_2_ACC_Y_range"],
                row["Deriv_2_ACC_Y_var"],
                row["Deriv_2_ACC_Y_std_dev"],
                row["Deriv_2_ACC_Y_abs_dev"],
                row["Deriv_2_ACC_Y_rms"],
                row["Deriv_2_ACC_Y_kurtosis"],
                row["Deriv_2_ACC_Y_skew"],
            ) = get_statistics(data=second_deriv_ACC_Y)
            # ACC wavelet features:
            (
                row["Wavelet_1Hz_ACC_Y_mean"],
                row["Wavelet_1Hz_ACC_Y_median"],
                row["Wavelet_1Hz_ACC_Y_min"],
                row["Wavelet_1Hz_ACC_Y_max"],
                row["Wavelet_1Hz_ACC_Y_max_amp"],
                row["Wavelet_1Hz_ACC_Y_range"],
                row["Wavelet_1Hz_ACC_Y_var"],
                row["Wavelet_1Hz_ACC_Y_std_dev"],
                row["Wavelet_1Hz_ACC_Y_abs_dev"],
                row["Wavelet_1Hz_ACC_Y_rms"],
                row["Wavelet_1Hz_ACC_Y_kurtosis"],
                row["Wavelet_1Hz_ACC_Y_skew"],
            ) = get_statistics(data=ACC_Y_cD_3)
            (
                row["Wavelet_2Hz_ACC_Y_mean"],
                row["Wavelet_2Hz_ACC_Y_median"],
                row["Wavelet_2Hz_ACC_Y_min"],
                row["Wavelet_2Hz_ACC_Y_max"],
                row["Wavelet_2Hz_ACC_Y_max_amp"],
                row["Wavelet_2Hz_ACC_Y_range"],
                row["Wavelet_2Hz_ACC_Y_var"],
                row["Wavelet_2Hz_ACC_Y_std_dev"],
                row["Wavelet_2Hz_ACC_Y_abs_dev"],
                row["Wavelet_2Hz_ACC_Y_rms"],
                row["Wavelet_2Hz_ACC_Y_kurtosis"],
                row["Wavelet_2Hz_ACC_Y_skew"],
            ) = get_statistics(data=ACC_Y_cD_2)
            (
                row["Wavelet_4Hz_ACC_Y_mean"],
                row["Wavelet_4Hz_ACC_Y_median"],
                row["Wavelet_4Hz_ACC_Y_min"],
                row["Wavelet_4Hz_ACC_Y_max"],
                row["Wavelet_4Hz_ACC_Y_max_amp"],
                row["Wavelet_4Hz_ACC_Y_range"],
                row["Wavelet_4Hz_ACC_Y_var"],
                row["Wavelet_4Hz_ACC_Y_std_dev"],
                row["Wavelet_4Hz_ACC_Y_abs_dev"],
                row["Wavelet_4Hz_ACC_Y_rms"],
                row["Wavelet_4Hz_ACC_Y_kurtosis"],
                row["Wavelet_4Hz_ACC_Y_skew"],
            ) = get_statistics(data=ACC_Y_cD_1)

            (
                row["ACC_Z_mean"],
                row["ACC_Z_median"],
                row["ACC_Z_min"],
                row["ACC_Z_max"],
                row["ACC_Z_max_amp"],
                row["ACC_Z_range"],
                row["ACC_Z_var"],
                row["ACC_Z_std_dev"],
                row["ACC_Z_abs_dev"],
                row["ACC_Z_rms"],
                row["ACC_Z_kurtosis"],
                row["ACC_Z_skew"],
            ) = get_statistics(data=acc_z_windows[i])
            (
                row["Deriv_ACC_Z_mean"],
                row["Deriv_ACC_Z_median"],
                row["Deriv_ACC_Z_min"],
                row["Deriv_ACC_Z_max"],
                row["Deriv_ACC_Z_max_amp"],
                row["Deriv_ACC_Z_range"],
                row["Deriv_ACC_Z_var"],
                row["Deriv_ACC_Z_std_dev"],
                row["Deriv_ACC_Z_abs_dev"],
                row["Deriv_ACC_Z_rms"],
                row["Deriv_ACC_Z_kurtosis"],
                row["Deriv_ACC_Z_skew"],
            ) = get_statistics(data=deriv_ACC_Z)
            (
                row["Deriv_2_ACC_Z_mean"],
                row["Deriv_2_ACC_Z_median"],
                row["Deriv_2_ACC_Z_min"],
                row["Deriv_2_ACC_Z_max"],
                row["Deriv_2_ACC_Z_max_amp"],
                row["Deriv_2_ACC_Z_range"],
                row["Deriv_2_ACC_Z_var"],
                row["Deriv_2_ACC_Z_std_dev"],
                row["Deriv_2_ACC_Z_abs_dev"],
                row["Deriv_2_ACC_Z_rms"],
                row["Deriv_2_ACC_Z_kurtosis"],
                row["Deriv_2_ACC_Z_skew"],
            ) = get_statistics(data=second_deriv_ACC_Z)
            # ACC wavelet features:
            (
                row["Wavelet_1Hz_ACC_Z_mean"],
                row["Wavelet_1Hz_ACC_Z_median"],
                row["Wavelet_1Hz_ACC_Z_min"],
                row["Wavelet_1Hz_ACC_Z_max"],
                row["Wavelet_1Hz_ACC_Z_max_amp"],
                row["Wavelet_1Hz_ACC_Z_range"],
                row["Wavelet_1Hz_ACC_Z_var"],
                row["Wavelet_1Hz_ACC_Z_std_dev"],
                row["Wavelet_1Hz_ACC_Z_abs_dev"],
                row["Wavelet_1Hz_ACC_Z_rms"],
                row["Wavelet_1Hz_ACC_Z_kurtosis"],
                row["Wavelet_1Hz_ACC_Z_skew"],
            ) = get_statistics(data=ACC_Z_cD_3)
            (
                row["Wavelet_2Hz_ACC_Z_mean"],
                row["Wavelet_2Hz_ACC_Z_median"],
                row["Wavelet_2Hz_ACC_Z_min"],
                row["Wavelet_2Hz_ACC_Z_max"],
                row["Wavelet_2Hz_ACC_Z_max_amp"],
                row["Wavelet_2Hz_ACC_Z_range"],
                row["Wavelet_2Hz_ACC_Z_var"],
                row["Wavelet_2Hz_ACC_Z_std_dev"],
                row["Wavelet_2Hz_ACC_Z_abs_dev"],
                row["Wavelet_2Hz_ACC_Z_rms"],
                row["Wavelet_2Hz_ACC_Z_kurtosis"],
                row["Wavelet_2Hz_ACC_Z_skew"],
            ) = get_statistics(data=ACC_Z_cD_2)
            (
                row["Wavelet_4Hz_ACC_Z_mean"],
                row["Wavelet_4Hz_ACC_Z_median"],
                row["Wavelet_4Hz_ACC_Z_min"],
                row["Wavelet_4Hz_ACC_Z_max"],
                row["Wavelet_4Hz_ACC_Z_max_amp"],
                row["Wavelet_4Hz_ACC_Z_range"],
                row["Wavelet_4Hz_ACC_Z_var"],
                row["Wavelet_4Hz_ACC_Z_std_dev"],
                row["Wavelet_4Hz_ACC_Z_abs_dev"],
                row["Wavelet_4Hz_ACC_Z_rms"],
                row["Wavelet_4Hz_ACC_Z_kurtosis"],
                row["Wavelet_4Hz_ACC_Z_skew"],
            ) = get_statistics(data=ACC_Z_cD_1)

            final_data.append(row)

# Create DataFrame once at the end
final_df = pd.DataFrame(final_data)
final_df

 Participants:   0%|          | 0/15 [00:00<?, ?it/s]

 Segments:   0%|          | 0/31 [00:00<?, ?it/s]

 Windows:   0%|          | 0/1705 [00:00<?, ?it/s]

 Windows:   0%|          | 0/9 [00:00<?, ?it/s]

 Windows:   0%|          | 0/3666 [00:00<?, ?it/s]

 Windows:   0%|          | 0/2401 [00:00<?, ?it/s]

 Windows:   0%|          | 0/1441 [00:00<?, ?it/s]